# ARO 3D Studio — real image-to-3D smoke test (Colab)
Runs the **real TripoSR engine through the Studio** (upload → batch generation job → raw asset → viewer preview → automatic
preparation → validation → verified GLB export) for **at least three genuinely different images**, and records evidence.

* Runtime ▸ Change runtime type ▸ **T4 GPU** (free tier is enough; first model download is ~1.7 GB).
* You need **one vehicle photo** and (optionally) other photos you own or that are licensed for this use. TripoSR's own `examples/` folder supplies non-vehicle objects.
* Nothing here is a pass until you read the evidence and look at the renders. Visual quality is judged by a person, not by this notebook.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU"

In [ ]:
BRANCH = "claude/zen-johnson-je3dzk"  # or the commit you are verifying
!git clone -b $BRANCH https://github.com/AffanNamu/ARO-3D-STUDIO /content/aro
%cd /content/aro
!git log -1 --format='%H %s'
!pip -q install -r requirements.txt

In [ ]:
import json, subprocess, os

lock = json.load(open("aro_studio/engines/locks/triposr.json"))
commit = lock["code"]["commit"]
if not os.path.isdir("/content/TripoSR"):
    !git clone https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
!git -C /content/TripoSR checkout $commit
# torch is already installed by Colab; requirements-triposr.txt asks for torch>=2.6.0 (CVE-2025-32434) and builds torchmcubes
!pip -q install -r requirements-triposr.txt
os.environ["PYTHONPATH"] = "/content/TripoSR"

In [ ]:
# Lock + weights: every file is SHA-256-checked against aro_studio/engines/locks/triposr.json BEFORE it is loaded.
!python -c "from aro_studio.engines.weights import load_lock; print('lock missing:', load_lock('triposr').missing())"
!python scripts/verify_triposr_weights.py

In [ ]:
# Khronos glTF validator for export verification (optional but recommended)
!mkdir -p /content/gv && cd /content/gv && npm init -y >/dev/null && npm i gltf-validator >/dev/null 2>&1
os.environ["ARO_GLTF_VALIDATOR_NODE_PATH"] = "/content/gv/node_modules"

In [ ]:
# 1) Upload your VEHICLE photo (and any other photos). Files land in /content.
from google.colab import files

up = files.upload()  # choose e.g. truck.jpg
print(list(up))

In [ ]:
# 2) Describe the images. EDIT this list. >= 3 items, at least one vehicle category and one non-vehicle category.
#    'source' records who owns the image / its licence. 'length_m' (real length along the front-back axis) is optional:
#    without it the prepared size is a flagged PLACEHOLDER. 'up'/'forward' are the model's native axes if you know them.
!ls /content/TripoSR/examples | head -40

In [ ]:
import json

VEHICLE_FILE = list(up)[0]  # the vehicle photo you uploaded
items = [
    {
        "path": f"/content/{VEHICLE_FILE}",
        "category": "heavy_truck",
        "name": "Vehicle photo",
        "source": "TODO: owner / licence",
    },
    {
        "path": "/content/TripoSR/examples/chair.png",
        "category": "furniture",
        "name": "Chair",
        "source": "TripoSR repo examples (MIT)",
    },
    {
        "path": "/content/TripoSR/examples/teapot.png",
        "category": "household_object",
        "name": "Teapot",
        "source": "TripoSR repo examples (MIT)",
    },
]
import os

for it in items:
    assert os.path.isfile(it["path"]), f"missing {it['path']} - pick a file from the listing above"
json.dump({"items": items}, open("/content/manifest.json", "w"), indent=1)

In [ ]:
# 3) The real smoke test through the Studio (force=true: a cache hit can never count). Takes a few minutes.
!python scripts/real_inference_smoke.py --manifest /content/manifest.json --out /content/aro/docs/acceptance/evidence/ai/smoke --preset BALANCED

In [ ]:
# 4) The gated real-model unit tests (loads the model, one inference, cache behaviour, Studio path with this manifest)
!ARO_RUN_MODEL_TESTS=1 ARO_SMOKE_MANIFEST=/content/manifest.json python -m pytest -m model -v -rs tests/test_triposr_real.py 2>&1 | tail -30

In [ ]:
# 5) LOOK at the renders, then fill in human_review in smoke-evidence.json (recognisable? defects? which way is front?)
import glob
from IPython.display import Image, display

for p in sorted(glob.glob("/content/aro/docs/acceptance/evidence/ai/smoke/views/*-raw-3q.png")):
    print(p)
    display(Image(p, width=320))
for p in sorted(glob.glob("/content/aro/docs/acceptance/evidence/ai/smoke/views/*-prepared-3q.png")):
    print(p)
    display(Image(p, width=320))

In [ ]:
# 6) Bundle the evidence + the (possibly changed) lock and download it
!cd /content/aro && zip -qr /content/aro-evidence.zip docs/acceptance/evidence/ai aro_studio/engines/locks/triposr.json
from google.colab import files

files.download("/content/aro-evidence.zip")